In [1]:
!python --version

Python 3.11.14


### Get OWL files

https://reactome.org/download/current/biopax/{pathway_id}_level3.owl

In [2]:
import sys
from pathlib import Path

from pybiopax import model_from_owl_file
from collections import Counter

ROOT0 = Path("/home/flavio/uv/perturb_agent/")
ROOT_SRC = ROOT0 / "src"

if str(ROOT_SRC) not in sys.path:
    sys.path.append(str(ROOT_SRC))

print("ROOT0:", ROOT0)
print("ROOT_SRC added:", ROOT_SRC)

from libs.Basic import *
from libs.dashcyto_lib import DASH_CYTO
from libs.reactome_lib import Reactome


ROOT0: /home/flavio/uv/perturb_agent
ROOT_SRC added: /home/flavio/uv/perturb_agent/src


In [3]:
rea = Reactome(ROOT0=ROOT0)

rea.root_reactome, rea.root_owl

(PosixPath('/home/flavio/uv/perturb_agent/data/reactome'),
 PosixPath('/home/flavio/uv/perturb_agent/owl'))

In [4]:
dfr = rea.open_reactome()
print(len(dfr))
dfr.head(3)

2119


,pathway,pathway_id
0,2-LTR Circle Formation,R-HSA-164843
1,Abacavir ADME,R-HSA-2161522
2,Abacavir Transmembrane Transport,R-HSA-2161517


### pybiopax

uv pip install pybiopax  
uv pip install ipywidgets  

In [5]:
hsa_model = rea.open_human_owl(force=False)

print(type(hsa_model))
print(len(hsa_model.objects))


Processing OWL elements:   0%|          | 0.00/545k [00:00<?, ?it/s]

<class 'pybiopax.biopax.model.BioPaxModel'>
544590


In [6]:
class_counts = Counter(type(obj).__name__ for obj in hsa_model.objects.values())

i=0
for cls, n in class_counts.most_common(20):
    print(cls, n)
    i+=1
    if i==10: break

UnificationXref 181879
SequenceSite 67532
Stoichiometry 43907
PublicationXref 39028
Protein 33650
FragmentFeature 30043
SequenceInterval 30043
PathwayStep 18863
Complex 17459
BiochemicalReaction 15994


In [7]:
pathways, pathway_index = rea.get_pathways_from_model()
print("Number of pathways:", len(pathways))
print("Number of pathway IDs:", len(pathway_index))


Number of pathways: 2870
Number of pathway IDs: 5740


In [8]:
dfo = rea.pathways_to_df(pathways)
dfo.head(12)

,pathway_object,rid,is_ID
0,<pybiopax.biopax.base.Pathway object at 0x75a2...,1852241,False
1,<pybiopax.biopax.base.Pathway object at 0x75a2...,R-HSA-1852241,True
2,<pybiopax.biopax.base.Pathway object at 0x75a2...,R-HSA-1592230,True
3,<pybiopax.biopax.base.Pathway object at 0x75a2...,1592230,False
4,<pybiopax.biopax.base.Pathway object at 0x75a2...,2151209,False
5,<pybiopax.biopax.base.Pathway object at 0x75a2...,R-HSA-2151209,True
6,<pybiopax.biopax.base.Pathway object at 0x75a2...,R-HSA-2151201,True
7,<pybiopax.biopax.base.Pathway object at 0x75a2...,2151201,False
8,<pybiopax.biopax.base.Pathway object at 0x75a2...,8949613,False
9,<pybiopax.biopax.base.Pathway object at 0x75a2...,R-HSA-8949613,True


In [9]:
for p in pathways[:3]:
    print(p.uid, getattr(p, "display_name", None), getattr(p, "pid", None)) 

Pathway1 Organelle biogenesis and maintenance None
Pathway2 Mitochondrial biogenesis None
Pathway3 Activation of PPARGC1A (PGC-1alpha) by phosphorylation None


In [10]:
i=0
for key, pathw in pathway_index.items():
    print(f"{key:15} {pathw}  {pathw.uid} {pathw.display_name}")
    i+=1
    if i==3: break

1852241         <pybiopax.biopax.base.Pathway object at 0x75a2345b8a90>  Pathway1 Organelle biogenesis and maintenance
R-HSA-1852241   <pybiopax.biopax.base.Pathway object at 0x75a2345b8a90>  Pathway1 Organelle biogenesis and maintenance
R-HSA-1592230   <pybiopax.biopax.base.Pathway object at 0x75a2345b82d0>  Pathway2 Mitochondrial biogenesis


In [11]:
pid = "R-HSA-164843"

p = pathway_index.get(pid)

if p is None:
    print("Not found")
else:
    print("Found:", p.display_name)

Found: 2-LTR circle formation


### Recursively collect all referenced objects

In [12]:
i=0
p = pathways[i]
print(f"Pathway {i}: {p.uid} - {p.display_name}")


sub_objects = rea.collect_referenced_objects(p, include_inverse=False)

print("Objects in pathway subset:", len(sub_objects))

Pathway 0: Pathway1 - Organelle biogenesis and maintenance
Objects in pathway subset: 62748


In [13]:
uid, pathway_id, pathway = rea.get_pathway_names(p, dfo)
uid, pathway_id, pathway

('Pathway1', 'R-HSA-1852241', 'Organelle biogenesis and maintenance')

In [14]:
verbose=True
force=False

ret = rea.save_owl_model(pathway_id, pathway, sub_objects=sub_objects, force=force, verbose=verbose)
ret

File /home/flavio/uv/perturb_agent/owl/R-HSA-1852241_level3.owl already exists.


True

### Next check is the OWL export function available in your installed pybiopax:

In [15]:
import pybiopax

print([x for x in dir(pybiopax) if "owl" in x.lower()])

['model_from_owl_file', 'model_from_owl_gz', 'model_from_owl_str', 'model_from_owl_url', 'model_to_owl_file', 'model_to_owl_str']


In [16]:
fname_owl = rea.fname_owl%(pathway_id)

filename = rea.root_owl / fname_owl
test_model = pybiopax.model_from_owl_file(filename)

print(type(test_model))
print(len(test_model.objects))

Processing OWL elements:   0%|          | 0.00/5.22k [00:00<?, ?it/s]

<class 'pybiopax.biopax.model.BioPaxModel'>
5221


### DASH_CYTO

In [ ]:
dcy = DASH_CYTO(ROOT0=ROOT0)

rdf = dcy.read_owl(pathway_id, pathway)
height = "1100px"
width = "100%"
marginTop="20px"

dcy.run_app(height=height, width=width, marginTop=marginTop)

Open in browser: http://localhost:8050
Dash app running on http://127.0.0.1:8050/


### Read and Plot any pathway

In [ ]:
dcy = DASH_CYTO(ROOT0=ROOT0)

pathway = 'Chaperone Mediated Autophagy'
pathway_id = 'R-HSA-9613829'

rdf = dcy.read_owl(pathway_id, pathway)
height = "1100px"
width = "100%"
marginTop="20px"

dcy.run_app(height=height, width=width, marginTop=marginTop)

Open in browser: http://localhost:8050
Dash app running on http://127.0.0.1:8050/
